# 01 - DataFrame Quickstart

`predict_dataframe()` is the recommended boundary for notebooks and application
integration. Input is long-form: one row per test day and one `TestId` per lactation.
It validates the group, converts normal decimal values to legacy Format-4 scaling,
runs the kernel, and returns one named output row per lactation.

In [1]:
from pathlib import Path


def find_repo_root() -> Path:
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "packages/models/bestpred").is_dir():
            return candidate
    raise RuntimeError("Run this notebook from a Bovi repository checkout")


ROOT = find_repo_root()
PACKAGE_ROOT = ROOT / "packages/models/bestpred"
FIXTURES = PACKAGE_ROOT / "tests/fixtures"
PARAMETERS = FIXTURES / "source11_current/bestpred.par"

## Load canonical test-day data

In [2]:
import pandas as pd

input_path = PACKAGE_ROOT / "notebooks/data/example_test_days.csv"
test_days = pd.read_csv(input_path, dtype={"BirthDate": str, "FreshDate": str})
test_days[["TestId", "DaysInMilk", "MilkingYield", "FatPercent", "ProteinPercent", "SCS"]]

,TestId,DaysInMilk,MilkingYield,FatPercent,ProteinPercent,SCS
0,cow-42-l2,30,70.0,3.9,3.2,2.1
1,cow-42-l2,60,75.0,3.8,3.1,2.2
2,cow-42-l2,120,67.0,4.0,3.3,2.4
3,cow-42-l2,200,58.0,4.1,3.4,2.6
4,cow-42-l2,280,46.0,4.2,3.5,2.8
5,cow-77-l1,20,62.0,3.7,3.1,2.0
6,cow-77-l1,55,69.0,3.8,3.2,2.1
7,cow-77-l1,110,64.0,3.9,3.3,2.3
8,cow-77-l1,190,53.0,4.1,3.4,2.5
9,cow-77-l1,270,41.0,4.3,3.6,2.7


Identity, dates, parity, target length, and herd 305-day baselines must remain constant
within a `TestId`. `DaysInMilk` must be unique. The example parameter file uses pounds
for input and output (`UNITSin='P'`, `UNITSout='P'`); percentages and SCS remain normal
decimal values at this high-level API.

In [3]:
from bestpred import dataframe_to_records, predict_dataframe

converted = dataframe_to_records(test_days)
pd.DataFrame(
    {
        "TestId": converted.test_ids,
        "Format4 cow": [record.cow_id for record in converted.records],
        "Test days": [len(record.segments) for record in converted.records],
        "Stored first milk value": [record.segments[0].milk_yield for record in converted.records],
    }
)

,TestId,Format4 cow,Test days,Stored first milk value
0,cow-42-l2,HCOW42,5,700
1,cow-77-l1,HCOW77,5,620


The stored milk value is ten times the supplied value because `Format4Record` is the
fixed-width compatibility boundary. Application code should normally avoid doing this
scaling itself.

In [4]:
predictions = predict_dataframe(test_days, PARAMETERS)
predictions[
    [
        "TestId",
        "MilkYield305",
        "FatYield305",
        "ProteinYield305",
        "SCSYield305",
        "MilkYieldReliability",
        "DCRMilk",
    ]
].round(2)

,TestId,MilkYield305,FatYield305,ProteinYield305,SCSYield305,MilkYieldReliability,DCRMilk
0,cow-42-l2,19372.68,761.40,624.60,77.32,0.75,78.31
1,cow-77-l1,20387.36,780.75,653.89,77.00,0.73,76.29


## Map caller-specific columns

In [5]:
caller_data = test_days.rename(columns={"DaysInMilk": "dim", "MilkingYield": "milk_lb"})
mapped = predict_dataframe(
    caller_data,
    PARAMETERS,
    column_map={"DaysInMilk": "dim", "MilkingYield": "milk_lb"},
)
mapped[["TestId", "MilkYield305", "DCRMilk"]].round(2)

,TestId,MilkYield305,DCRMilk
0,cow-42-l2,19372.68,78.31
1,cow-77-l1,20387.36,76.29


Optional fields receive documented defaults. In particular, omitting all component
columns sets `TimesSampled=0`; supplying component columns defaults it to `2`. A measured
zero should be supplied explicitly rather than represented by an absent column.

## Validation failures are explicit

In [6]:
examples = {}

try:
    dataframe_to_records(test_days.drop(columns="HerdMilk305"))
except ValueError as exc:
    examples["missing column"] = str(exc)

duplicate_dim = pd.concat([test_days, test_days.iloc[[0]]], ignore_index=True)
try:
    dataframe_to_records(duplicate_dim)
except ValueError as exc:
    examples["duplicate DIM"] = str(exc)

inconsistent = test_days.copy()
inconsistent.loc[1, "Parity"] = 3
try:
    dataframe_to_records(inconsistent)
except ValueError as exc:
    examples["inconsistent group"] = str(exc)

pd.Series(examples, name="Message").to_frame()

,Message
missing column,Missing required BESTPRED DataFrame columns: H...
duplicate DIM,TestId 'cow-42-l2' contains duplicate DaysInMi...
inconsistent group,Parity must be constant within TestId 'cow-42-l2'


The complete required/optional column tables, defaults, units, and all named output
columns are maintained in the package README. Treat `bestpred.par` as model configuration:
its unit settings and numerical options are part of prediction provenance.